# OWL 2 DL Reasoning with HermiT — User Guide

The [Graph Inferencing guide](02b-graphs-inferencing.ipynb) covers RDFS and OWL 2 RL reasoning via `infer()`, both backed by `owlrl`, a fast Python rule engine. This guide covers a third, separate `infer()` profile: `profile="owl-dl"`, genuine OWL 2 DL reasoning via [HermiT](http://www.hermit-reasoner.com/) through the `owlready2` library.

HermiT is a *tableau* reasoner: rather than applying a fixed set of if-then rules, it tries to actually construct a model — a concrete interpretation that makes every axiom true at once based on the data in the graph. Where an axiom leaves a choice open (a class defined as a union, an existential restriction, and so on), HermiT picks a branch and continues; if that choice leads to a contradiction elsewhere in the data, it discards that branch and backtracks to try another. It keeps branching and backtracking over every open choice until it either succeeds — the data is consistent — or exhausts every possibility, in which case the data is inconsistent. 

If HermiT finds a triple that must be true in every possible model it could build, that triple is inferred. 

This systematic exploration is what makes it sound and complete for the full OWL 2 DL profile, rather than the restricted, rule-friendly OWL 2 RL fragment `owlrl` implements.

Section 1 (with 1.a, 1.b, and 1.c, three more elaborate variants) demonstrates entailments this makes possible that `owlrl` cannot derive (1.b is an exception - see its own note); section 2 demonstrates a genuinely different kind of question, whether a *class* can ever have any members at all; section 3 demonstrates an inconsistency HermiT correctly detects that passes through `owlrl` silently.

This is a separate guide from `02b` because the prerequisites are meaningfully heavier: a real Java runtime, not just a pip install.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

`owlready2`additionally requires a Java runtime on `PATH`. HermiT is a Java program. If java is not found or not configured correctly, an error message is raised.  

Run cells from top to bottom.

In [1]:
from starlayer import StarLayerGraph, Namespace, RDF

EX = Namespace("http://example.org/")

## 1. OWL 2 DL entailment

`g.infer(profile="owl-dl")` runs genuine OWL 2 DL reasoning. 

### 1.a A nested, three-way disjunction

OWL 2 DL can handle complex reasoning. When an anlysis identifies a triple that must be true in any possible scenario that triple can be inferred.

In [2]:
# mycar is a Car and known NOT to be FossilFuel (itself Gas-or-Diesel) - ruling
# out that whole branch leaves only Electric, even though Electric is never
# asserted directly and Gas/Diesel are never even individually ruled out.
g_nested = StarLayerGraph()
g_nested.bind("ex", EX)
g_nested.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Gas owl:disjointWith ex:Electric, ex:Diesel .
    ex:Diesel owl:disjointWith ex:Electric .
    ex:FossilFuel owl:equivalentClass [ owl:unionOf ( ex:Gas ex:Diesel ) ] .
    ex:Car rdfs:subClassOf [ owl:unionOf ( ex:FossilFuel ex:Electric ) ] .
    ex:mycar a ex:Car, [ owl:complementOf ex:FossilFuel ] .
""", format="turtle12")

print("Before")
print(g_nested.serialize(format="turtle12"))

closed_nested = g_nested.infer(profile="owl-dl")
print("mycar a Electric:", (EX.mycar, RDF.type, EX.Electric) in closed_nested)
print("mycar a Gas:      ", (EX.mycar, RDF.type, EX.Gas) in closed_nested)
print("mycar a Diesel:   ", (EX.mycar, RDF.type, EX.Diesel) in closed_nested)

Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Car rdfs:subClassOf [
        owl:unionOf ( ex:FossilFuel ex:Electric )
    ] .

ex:Diesel owl:disjointWith ex:Electric .

ex:FossilFuel owl:equivalentClass [
        owl:unionOf ( ex:Gas ex:Diesel )
    ] .

ex:Gas owl:disjointWith ex:Diesel, ex:Electric .

ex:mycar a [
        owl:complementOf ex:FossilFuel
    ], ex:Car .



mycar a Electric: True
mycar a Gas:       False
mycar a Diesel:    False


### 1.b A universal restriction (`allValuesFrom`)

A different construct: `ex:ecocar` requires *all* of its `ex:hasBattery` fillers to be `ex:LithiumBattery` (a universal restriction, `owl:allValuesFrom`); `ex:battery1` is known to be one of `ex:ecocar`'s batteries, and known to be *a* `ex:Battery` (itself `ex:LithiumBattery`-or-`ex:LeadAcidBattery`). The restriction rules out the `LeadAcidBattery` branch, so `battery1` must be `LithiumBattery`.

Worth knowing: `owlrl` has its own direct rule for `allValuesFrom` and derives this one too - unlike sections 1/1.a/2, this specific case isn't `owl-dl`-exclusive, just a different construct worth seeing. What *is* worth being careful about: this example silently returned the wrong (empty) answer the first time it was written, because `ex:hasBattery` was never explicitly declared `a owl:ObjectProperty`. HermiT needs a property's kind (object vs. datatype) explicitly stated to resolve a restriction on it correctly - `owlrl` happened to tolerate the same omission, which is exactly why the bug went unnoticed until this was checked directly against HermiT's own output.

In [3]:
# ecocar requires ALL its batteries to be LithiumBattery; battery1 is known
# to be A Battery (Lithium-or-LeadAcid) and known to be one of ecocar's
# batteries - the restriction rules out LeadAcidBattery for this one filler.
g_avf = StarLayerGraph()
g_avf.bind("ex", EX)
g_avf.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:hasBattery a owl:ObjectProperty .
    ex:LithiumBattery owl:disjointWith ex:LeadAcidBattery .
    ex:Battery rdfs:subClassOf [ owl:unionOf ( ex:LithiumBattery ex:LeadAcidBattery ) ] .
    ex:ecocar a [ a owl:Restriction ; owl:onProperty ex:hasBattery ; owl:allValuesFrom ex:LithiumBattery ] .
    ex:ecocar ex:hasBattery ex:battery1 .
    ex:battery1 a ex:Battery .
""", format="turtle12")

print("Before")
print(g_avf.serialize(format="turtle12"))

closed_avf = g_avf.infer(profile="owl-dl")
print("battery1 a LithiumBattery:", (EX.battery1, RDF.type, EX.LithiumBattery) in closed_avf)
print("battery1 a LeadAcidBattery:", (EX.battery1, RDF.type, EX.LeadAcidBattery) in closed_avf)

Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Battery rdfs:subClassOf [
        owl:unionOf ( ex:LithiumBattery ex:LeadAcidBattery )
    ] .

ex:LithiumBattery owl:disjointWith ex:LeadAcidBattery .

ex:battery1 a ex:Battery .

ex:ecocar ex:hasBattery ex:battery1 ;
    a [
        a owl:Restriction ;
        owl:allValuesFrom ex:LithiumBattery ;
        owl:onProperty ex:hasBattery
    ] .

ex:hasBattery a owl:ObjectProperty .



battery1 a LithiumBattery: True
battery1 a LeadAcidBattery: False


### 1.c A small logic puzzle

OWL dl-reasoning can solve complex logic problems. 

Three people, three houses, three jobs, solved from a positive clue, a negative clue, and three general rules — the classic "logic grid" pattern.

**Setting it up properly**, `ex:livesIn` and `ex:hasJob` are each declared both `owl:FunctionalProperty` and `owl:InverseFunctionalProperty`. `ex:Person` requires *some* `ex:livesIn` value from the nominal set `{RedHouse, BlueHouse, GreenHouse}` (`owl:oneOf` + `owl:someValuesFrom`), and likewise for `ex:hasJob` over `{Doctor, Teacher, Engineer}`. Because that nominal set has exactly three members, and `ex:alice`, `ex:bob`, `ex:carol` are declared pairwise `owl:differentFrom`, an injective mapping from three distinct people into a three-element set is automatically a bijection — every house and every job ends up filled by exactly one person, without ever asserting that directly.

**The clues** are then genuinely minimal:
- *(positive)* `ex:bob` lives in the Red House.
- *(negative)* `ex:carol` is not a Doctor.
- *(general rule)* whoever lives in the Red House is not a Doctor.
- *(general rule)* the Doctor does not live in the Green House.
- *(general rule)* whoever lives in the Green House is not an Engineer.

Each "rule" is a real subsumption between two restrictions (`[...] rdfs:subClassOf [ owl:complementOf [...] ]`), not a one-off fact about a specific individual — it applies to whoever ends up satisfying its antecedent, however that gets determined.

**Solving it**: rule 1 plus Bob's Red House rules Bob out for Doctor; the negative clue rules Carol out too — since the job mapping is a bijection over exactly three people, Alice must be the Doctor. Rule 2 then rules out the Green House for Alice (she's the Doctor); with Bob already in the Red House, the house bijection leaves Alice the Blue House and Carol the Green House. Rule 3 then rules out Engineer for Carol (she's in the Green House); with Alice already the Doctor, the job bijection leaves Carol as Teacher and Bob as Engineer.

In [10]:
g_puzzle = StarLayerGraph()
g_puzzle.bind("ex", EX)
g_puzzle.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

    # each person lives in at most one house and each house has a most one occupant.
    ex:livesIn a owl:ObjectProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty .

    # each person has at most one job and each job has a most one person.
    ex:hasJob a owl:ObjectProperty, owl:FunctionalProperty, owl:InverseFunctionalProperty .

    # a pserson lives in one of three houses
    # a person has one of three jobs
    ex:Person rdfs:subClassOf
        [ a owl:Restriction ; owl:onProperty ex:livesIn ;
          owl:someValuesFrom [ owl:oneOf ( ex:RedHouse ex:BlueHouse ex:GreenHouse ) ] ] ,
        [ a owl:Restriction ; owl:onProperty ex:hasJob ;
          owl:someValuesFrom [ owl:oneOf ( ex:Doctor ex:Teacher ex:Engineer ) ] ] .

                 
    ex:alice a ex:Person .
    ex:bob a ex:Person .
    ex:carol a ex:Person .
    ex:alice owl:differentFrom ex:bob, ex:carol .
    ex:bob owl:differentFrom ex:carol .

    # clue 1 (positive): Bob lives in the Red House
    ex:bob ex:livesIn ex:RedHouse .

    # clue 2 (negative): Carol is not a Doctor
    # Carol is a member of the class of things that are not Doctors.
    ex:carol a [ owl:complementOf [ a owl:Restriction ; owl:onProperty ex:hasJob ; owl:hasValue ex:Doctor ] ] .

    # rule 1: whoever lives in the Red House is not a Doctor
    [ a owl:Restriction ; owl:onProperty ex:livesIn ; owl:hasValue ex:RedHouse ]
        rdfs:subClassOf [ owl:complementOf [ a owl:Restriction ; owl:onProperty ex:hasJob ; owl:hasValue ex:Doctor ] ] .

    # rule 2: the Doctor does not live in the Green House
    [ a owl:Restriction ; owl:onProperty ex:hasJob ; owl:hasValue ex:Doctor ]
        rdfs:subClassOf [ owl:complementOf [ a owl:Restriction ; owl:onProperty ex:livesIn ; owl:hasValue ex:GreenHouse ] ] .

    # rule 3: whoever lives in the Green House is not an Engineer
    [ a owl:Restriction ; owl:onProperty ex:livesIn ; owl:hasValue ex:GreenHouse ]
        rdfs:subClassOf [ owl:complementOf [ a owl:Restriction ; owl:onProperty ex:hasJob ; owl:hasValue ex:Engineer ] ] .
""", format="turtle12")

closed_puzzle = g_puzzle.infer(profile="owl-dl")
print("Solution:")
for person in ("alice", "bob", "carol"):
    p = EX[person]
    house = [str(o).rsplit("/", 1)[-1] for o in closed_puzzle.objects(p, EX.livesIn)]
    job = [str(o).rsplit("/", 1)[-1] for o in closed_puzzle.objects(p, EX.hasJob)]
    print(f"  {person}: house={house}, job={job}")

Solution:
  alice: house=['BlueHouse'], job=['Doctor']
  bob: house=['RedHouse'], job=['Engineer']
  carol: house=['GreenHouse'], job=['Teacher']


## 2. Class unsatisfiability

A different kind of question than sections 1/1.a/1.b/1.c, which all ask "what type is this individual?" — here the question is about a *class itself*, with no individual involved at all: can anything possibly belong to it? `ex:ElectricOnly` is defined to require *all* its `ex:hasEngine` fillers to be `ex:ElectricMotor`, and *some* `ex:hasEngine` filler to be `ex:GasEngine` — but `ElectricMotor` and `GasEngine` are disjoint, so that required filler would have to be both at once. No individual could ever satisfy both restrictions simultaneously, so the class itself is empty — logically equivalent to `owl:Nothing`. Proving a class is unsatisfiable like this needs the same kind of exhaustive exploration as section 1's variants; `owlrl`'s forward-chaining rules have no way to conclude "nothing could ever satisfy this" in general.

In [5]:
from rdflib.namespace import OWL

# ElectricOnly requires ALL hasEngine fillers to be ElectricMotor, AND at
# least one hasEngine filler to be GasEngine - but the two are disjoint, so
# no individual could ever satisfy both restrictions at once.
g_unsat = StarLayerGraph()
g_unsat.bind("ex", EX)
g_unsat.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:hasEngine a owl:ObjectProperty .
    ex:ElectricMotor owl:disjointWith ex:GasEngine .
    ex:ElectricOnly rdfs:subClassOf
        [ a owl:Restriction ; owl:onProperty ex:hasEngine ; owl:allValuesFrom ex:ElectricMotor ] ,
        [ a owl:Restriction ; owl:onProperty ex:hasEngine ; owl:someValuesFrom ex:GasEngine ] .
""", format="turtle12")

print("Before")
print(g_unsat.serialize(format="turtle12"))

closed_unsat = g_unsat.infer(profile="owl-dl")
print("ElectricOnly is unsatisfiable (== owl:Nothing):",
      (EX.ElectricOnly, OWL.equivalentClass, OWL.Nothing) in closed_unsat)

# owlrl, for contrast, never concludes this
delta_rl = g_unsat.infer(profile="owl-rl", mode="delta")
print("owlrl also concludes this:",
      (EX.ElectricOnly, OWL.equivalentClass, OWL.Nothing) in delta_rl)

Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:ElectricMotor owl:disjointWith ex:GasEngine .

ex:ElectricOnly rdfs:subClassOf [
        a owl:Restriction ;
        owl:onProperty ex:hasEngine ;
        owl:someValuesFrom ex:GasEngine
    ], [
        a owl:Restriction ;
        owl:allValuesFrom ex:ElectricMotor ;
        owl:onProperty ex:hasEngine
    ] .

ex:hasEngine a owl:ObjectProperty .



ElectricOnly is unsatisfiable (== owl:Nothing): True
owlrl also concludes this: False


## 3. Detecting inconsistencies

Every car must be electric or gas-powered; `ex:mycar` is asserted to be neither — a genuine contradiction, but not one of `owlrl`'s fixed local violation patterns (Inferencing guide [section 2.4](02b-graphs-inferencing.ipynb#2.4-Detecting-inconsistencies)), so it passes through `owlrl` silently. `profile="owl-dl"`'s consistency check is sound and complete, and raises `starlayergraph.graph.owl_dl.InconsistentOntologyError` outright rather than leaving it for you to notice.

In [6]:
from starlayergraph.graph.owl_dl import InconsistentOntologyError

# every car must be electric or gas-powered, but mycar is asserted to be
# neither - a genuine contradiction no single local rule pattern-matches
DATA = """
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Electric owl:disjointWith ex:Gas .
    ex:Car rdfs:subClassOf [ owl:unionOf ( ex:Electric ex:Gas ) ] .
    ex:mycar a ex:Car, [ owl:complementOf ex:Electric ], [ owl:complementOf ex:Gas ] .
"""

ERR = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")
g2 = StarLayerGraph()
g2.bind("ex", EX)
g2.parse(data=DATA, format="turtle12")
closed = g2.infer(profile="owl-rl")
print("owlrl flags this as an inconsistency:", bool(list(closed.objects(None, ERR.error))))

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.parse(data=DATA, format="turtle12")
try:
    g3.infer(profile="owl-dl")
    print("owl-dl: no exception raised - not reached")
except InconsistentOntologyError as e:
    print("owl-dl: InconsistentOntologyError raised:", e)

owlrl flags this as an inconsistency: False


owl-dl: InconsistentOntologyError raised: infer(profile='owl-dl') found self's data logically inconsistent under OWL 2 DL semantics.


## Where to go next

1. **[Getting Started](01-getting-started.ipynb)**
2. **[Graphs](02-graphs.ipynb)**
   - 2.b **[Inferencing](02b-graphs-inferencing.ipynb)** — RDFS/OWL-RL reasoning via `owlrl`, including `StarLayerGraph.infer()`.
3. **[SPARQL](03-sparql.ipynb)**
   - 3.b **[SPARQL inferencing](03b-sparql-inferencing.ipynb)** — query-time `entailment="rdfs"|"owl-rl"`. Query-time OWL 2 DL reasoning (`entailment="direct"`, mirroring `entailment="owl-rl"`) is not yet built - see that guide's own regime table.
5. **Other**
   - 5.e **OWL 2 DL reasoning with HermiT** — this guide.